# Customer Churn Prediction: EDA & Business Insights

This notebook documents the first analytical pass through the IBM Telco Customer Churn dataset. It covers data inspection, cleaning checks, churn distribution, and business segments that can inform retention strategy.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Load Dataset

In [ ]:
DATA_PATH = Path("../data/raw/telco_customer_churn.csv")
df = pd.read_csv(DATA_PATH)

## 3. Initial Data Checks

These checks show the size, structure, sample records, missing values, and target distribution.

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts(normalize=True).mul(100).round(2)

## 4. Data Cleaning Checks

`TotalCharges` is stored as text in the raw dataset. It needs to be converted to numeric before modeling.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].isnull().sum()

In [ ]:
df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head(20)

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["ChurnFlag"] = df["Churn"].map({"No": 0, "Yes": 1})

df[["TotalCharges", "ChurnFlag"]].isnull().sum()

## 5. Churn Distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn", hue="Churn", palette=["#2563eb", "#dc2626"], legend=False)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customer Count")
plt.show()

## 6. Business Segment Analysis

In [ ]:
segment_churn = (
    df.groupby("Contract")["ChurnFlag"]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
)

segment_churn

In [ ]:
plt.figure(figsize=(7, 4))
segment_churn.plot(kind="bar", color="#2563eb")
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
payment_churn = (
    df.groupby("PaymentMethod")["ChurnFlag"]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
)

payment_churn

In [ ]:
plt.figure(figsize=(8, 4))
payment_churn.plot(kind="bar", color="#059669")
plt.title("Churn Rate by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=30, ha="right")
plt.show()

In [ ]:
internet_churn = (
    df.groupby("InternetService")["ChurnFlag"]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
)

internet_churn

## 7. Tenure and Monthly Charges

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x="tenure", hue="Churn", fill=True, common_norm=False, alpha=0.35)
plt.title("Tenure Distribution by Churn Outcome")
plt.xlabel("Tenure in Months")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", hue="Churn", palette=["#2563eb", "#dc2626"], legend=False)
plt.title("Monthly Charges by Churn Outcome")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()

## 8. Business Readout

Key early findings:

- Churn is not evenly distributed across customers.
- Month-to-month contracts show materially higher churn than one-year or two-year contracts.
- Electronic check customers have elevated churn risk.
- Short-tenure customers are more likely to leave, which suggests onboarding and early-life engagement matter.
- These insights support a retention strategy focused on contract upgrades, payment friction reduction, and early customer success outreach.